# Task 6 — Context-Aware Chat Assistant with Memory

A chat assistant that remembers earlier turns of the conversation and
injects them as context into every new prompt, so it can answer questions
like "What is my name?" after the user previously said "My name is Ahmed."

## Objectives checklist
- [x] Load an LLM
- [x] Build a chat loop where the user can ask multiple questions
- [x] Store conversation history
- [x] Inject the previous conversation as context in each new prompt
- [x] Run a continuous chat loop and observe how memory affects responses
- [x] Automated tests proving memory actually affects responses

## About `USE_LIVE_MODEL`
Set the flag below to `True` to download and use a real Hugging Face model
(`google/flan-t5-base`) -- requires internet access. Left `False`, the
notebook uses a small rule-based offline stand-in model so the memory/context
logic can be demonstrated and verified without downloading anything, which
is what was used to produce the outputs saved in this notebook.

In [1]:
%pip install -q transformers torch

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

In [2]:
USE_LIVE_MODEL = False  # Set True to use a real Hugging Face LLM (requires internet)


## 2. Conversation memory

Stores every user/assistant turn and renders it as a plain-text transcript, trimmed to the last `max_turns` turns.

In [3]:
class ConversationMemory:
    def __init__(self, max_turns: int = 20):
        self.max_turns = max_turns
        self.history = []  # list of {"role": "user"|"assistant", "content": str}

    def add_user_message(self, message: str):
        self.history.append({"role": "user", "content": message})
        self._trim()

    def add_ai_message(self, message: str):
        self.history.append({"role": "assistant", "content": message})
        self._trim()

    def _trim(self):
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-self.max_turns * 2:]

    def build_context(self) -> str:
        lines = []
        for turn in self.history:
            speaker = "User" if turn["role"] == "user" else "Assistant"
            content = turn["content"]
            lines.append(f"{speaker}: {content}")
        return "\n".join(lines)

    def clear(self):
        self.history = []


print("ConversationMemory ready.")

ConversationMemory ready.


## 3. Generator

When `USE_LIVE_MODEL = True`, this loads `google/flan-t5-base`. Otherwise it uses a small rule-based offline model that can only answer from the prompt text it receives -- which makes it easy to prove memory is genuinely being injected (a model with no memory of its own can only get the name right if the conversation history was in the prompt).

In [4]:
import re


def offline_llm(prompt: str) -> str:
    """Tiny rule-based 'model': knows only what is written in the prompt it receives.

    Only the CURRENT question (the text after the last "User:") decides intent,
    since earlier turns stay in the prompt as history and could otherwise be
    mistaken for the current question on every later turn.
    """
    current_question = prompt.rsplit("User:", 1)[-1].split("\nAssistant:")[0].strip()
    if "what is my name" in current_question.lower():
        match = re.search(r"my name is (\w+)", prompt, re.IGNORECASE)
        return f"Your name is {match.group(1)}." if match else "I don\'t know your name yet."
    if "what do i like" in current_question.lower():
        match = re.search(r"i like (\w+)", prompt, re.IGNORECASE)
        return f"You like {match.group(1)}." if match else "I don\'t know what you like yet."
    match = re.search(r"my name is (\w+)", current_question, re.IGNORECASE)
    if match:
        return f"Nice to meet you, {match.group(1)}!"
    match = re.search(r"i like (\w+)", current_question, re.IGNORECASE)
    if match:
        return f"Got it, you like {match.group(1)}."
    return "Okay, tell me more."


def build_live_generator(model_name: str = "google/flan-t5-base", max_new_tokens: int = 128):
    from transformers import pipeline
    pipe = pipeline("text2text-generation", model=model_name, max_new_tokens=max_new_tokens)
    return lambda prompt: pipe(prompt)[0]["generated_text"].strip()


def get_generator():
    return build_live_generator() if USE_LIVE_MODEL else offline_llm


print(f"Generator configured (USE_LIVE_MODEL={USE_LIVE_MODEL}).")

Generator configured (USE_LIVE_MODEL=False).


## 4. Chat assistant

Builds a prompt from `[system instructions] + [full history] + [new question]`, sends it to the generator, then appends the exchange back into memory.

In [5]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. Use the conversation history below to "
    "answer the user\'s latest question, remembering any facts they told you."
)


class ChatAssistant:
    def __init__(self, generator=None):
        self.generator = generator or get_generator()
        self.memory = ConversationMemory()

    def build_prompt(self, user_message: str) -> str:
        context = self.memory.build_context()
        parts = [SYSTEM_PROMPT]
        if context:
            parts.append("Conversation so far:\n" + context)
        parts.append(f"User: {user_message}\nAssistant:")
        return "\n\n".join(parts)

    def ask(self, user_message: str) -> str:
        prompt = self.build_prompt(user_message)
        response = self.generator(prompt)
        self.memory.add_user_message(user_message)
        self.memory.add_ai_message(response)
        return response


print("ChatAssistant ready.")

ChatAssistant ready.


## 5. Demo run

Scripted conversation demonstrating the exact scenario from the task brief.

In [6]:
assistant = ChatAssistant()

demo_turns = [
    "My name is Ahmed.",
    "What is my name?",
    "I like hiking.",
    "What do I like?",
]

for turn in demo_turns:
    reply = assistant.ask(turn)
    print(f"You: {turn}")
    print(f"Assistant: {reply}\n")

You: My name is Ahmed.
Assistant: Nice to meet you, Ahmed!

You: What is my name?
Assistant: Your name is Ahmed.

You: I like hiking.
Assistant: Got it, you like hiking.

You: What do I like?
Assistant: You like hiking.



## 6. Automated tests (offline)

A fake generator that can only answer from the prompt text it receives proves the conversation history is genuinely being built and injected on each turn.

In [7]:
def test_memory_stores_turns():
    memory = ConversationMemory()
    memory.add_user_message("Hello")
    memory.add_ai_message("Hi there!")
    context = memory.build_context()
    assert "User: Hello" in context
    assert "Assistant: Hi there!" in context
    print("PASS test_memory_stores_turns")


def test_memory_trims_to_max_turns():
    memory = ConversationMemory(max_turns=2)
    for i in range(5):
        memory.add_user_message(f"msg {i}")
        memory.add_ai_message(f"reply {i}")
    assert len(memory.history) == 4
    print("PASS test_memory_trims_to_max_turns")


def test_assistant_remembers_name_across_turns():
    a = ChatAssistant(generator=offline_llm)
    first = a.ask("My name is Sara.")
    assert "Sara" in first
    second = a.ask("What is my name?")
    assert "Sara" in second, f"Expected the assistant to recall the name, got: {second!r}"
    print("PASS test_assistant_remembers_name_across_turns")


def test_prompt_without_history_has_no_prior_context():
    a = ChatAssistant(generator=offline_llm)
    prompt = a.build_prompt("What is my name?")
    assert "Conversation so far" not in prompt
    print("PASS test_prompt_without_history_has_no_prior_context")


def test_prompt_includes_full_history():
    a = ChatAssistant(generator=offline_llm)
    a.ask("My name is Layla.")
    prompt = a.build_prompt("What is my name?")
    assert "Conversation so far" in prompt
    assert "Layla" in prompt
    print("PASS test_prompt_includes_full_history")


def test_clear_memory_forgets_context():
    a = ChatAssistant(generator=offline_llm)
    a.ask("My name is Omar.")
    a.memory.clear()
    reply = a.ask("What is my name?")
    assert "don\'t know" in reply.lower()
    print("PASS test_clear_memory_forgets_context")


test_memory_stores_turns()
test_memory_trims_to_max_turns()
test_assistant_remembers_name_across_turns()
test_prompt_without_history_has_no_prior_context()
test_prompt_includes_full_history()
test_clear_memory_forgets_context()
print("\nAll chat assistant tests passed.")

PASS test_memory_stores_turns
PASS test_memory_trims_to_max_turns
PASS test_assistant_remembers_name_across_turns
PASS test_prompt_without_history_has_no_prior_context
PASS test_prompt_includes_full_history
PASS test_clear_memory_forgets_context

All chat assistant tests passed.


## 7. Interactive mode (optional)

Run this cell in a real Jupyter session to chat freely. Type `clear` to reset memory, `exit` to stop. Exits gracefully if there is no input available (e.g. run non-interactively).

In [8]:
interactive_assistant = ChatAssistant()
try:
    while True:
        user_message = input("You: ").strip()
        if user_message.lower() in ("exit", "quit"):
            break
        if user_message.lower() == "clear":
            interactive_assistant.memory.clear()
            print("Memory cleared.\n")
            continue
        if not user_message:
            continue
        print(f"Assistant: {interactive_assistant.ask(user_message)}\n")
except Exception:
    print("(No interactive input available -- skipping interactive mode.)")

(No interactive input available -- skipping interactive mode.)


## Notes
- Swap `get_generator()` for a larger/better instruction-tuned model, or any `callable(prompt) -> str` (e.g. an API-based chat model) -- the memory logic stays the same.
- `ConversationMemory(max_turns=...)` caps how many turns are kept in context to control prompt length.
